# EMNLP 4-Pager Data Analysis

Single notebook for all analysis phases (0–10) supporting the EMNLP 4-pager on topic domination beyond scalar and Pareto toxicity.

- **Primary run:** `data/outputs/20260211_2122`
- **Results:** `experiments/emnlp_data_analysis/results/`
- **Plan:** `scripts/EMNLP_4PAGER_PLAN.txt`

| Phase | Description |
|-------|-------------|
| 0 | Smoke gate — `smoke_gate.py` (offline) |
| 1 | Unified table — `build_unified_dataset.py` (offline) |
| 2 | Global + local Pareto ranks (species / reserves / archive) + pymoo PCP |
| 3 | Topic domination (TDI) — Google & OpenAI + rank agreement |
| 4 | Temporal species dynamics (EvolutionTracker, Fig 1A) |
| 5 | Counterfactual retention (Fig 3) — dual evaluator |
| 6 | Behaviour: UMAP, topic×axis heatmaps, cluster quality, d_g/d_p |
| 7 | Inference: permutation test, rank contingency, axis correlation |
| 8 | Sensitivity: ε-sweep, thresholds, population ablations |
| 9 | Cross-evaluator robustness (Jaccard + hierarchy) |
| 11 | Deep advanced analytics (single-axis vs MO vs speciated oracle, NSI) |
| 12 | Semantic topic labels and redacted exemplars (TF-IDF + cached LLM) |
| 10 | Manifest, results snippet, validation checklist, figure index |


## Setup

In [8]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def find_analysis_root() -> Path:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    for base in candidates:
        root = (
            base
            if (base / "analysis_utils.py").is_file()
            else base / "experiments" / "emnlp_data_analysis"
        )
        if (root / "analysis_utils.py").is_file():
            return root
    raise RuntimeError(
        "Could not locate experiments/emnlp_data_analysis/analysis_utils.py"
    )


ANALYSIS_ROOT = find_analysis_root()
REPO_ROOT = ANALYSIS_ROOT.parents[1]
RESULTS_DIR = ANALYSIS_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ANALYSIS_ROOT))

import importlib
import analysis_utils

importlib.reload(analysis_utils)
from analysis_utils import (
    DEFAULT_PRIMARY_RUN,
    OPENAI_AXIS_ORDER,
    PERSPECTIVE_AXIS_ORDER,
    annotate_pareto_ranks,
    load_unified_genomes,
    run_id_from_path,
    save_phase2_artifacts,
    save_phase3_artifacts,
    save_phase4_artifacts,
    save_phase5_artifacts,
    save_phase6_artifacts,
    save_phase7_artifacts,
    save_phase8_artifacts,
    save_phase9_artifacts,
    save_phase10_artifacts,
    run_all_remaining_phases,
    save_unified_artifacts,
    load_unified_from_artifacts,
)

# --- configurable ---
RUN_PATH = DEFAULT_PRIMARY_RUN
MIN_GENOMES = 5_000
MIN_TOPICS = 5
MIN_TOPIC_SIZE = 5
MIN_OBJECTIVE_FRAC = 0.95

print(f"Analysis root: {ANALYSIS_ROOT}")
print(f"Repo root:     {REPO_ROOT}")
print(f"Primary run:   {RUN_PATH}")
print(f"Results dir:   {RESULTS_DIR}")
print(f"Axis order:    {PERSPECTIVE_AXIS_ORDER}")


Analysis root: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis
Repo root:     /Users/onkars/Documents/Projects/ToxSearch-S
Primary run:   /Users/onkars/Documents/Projects/ToxSearch-S/data/outputs/20260211_2122
Results dir:   /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis/results
Axis order:    ['toxicity', 'severe_toxicity', 'identity_attack', 'insult', 'profanity', 'threat', 'sexually_explicit', 'flirtation']


## Phase 0 — Smoke Gate (offline)

Run **`smoke_gate.py`** before building the unified dataset. Local checks only (no OpenAI).

```bash
python experiments/emnlp_data_analysis/smoke_gate.py
```

**Output:** `results/gate0_smoke.json` (must be PASS).

| Check | Threshold |
|-------|-----------|
| Unified genome count |G| | ≥ 5,000 |
| Topics with ≥5 genomes | ≥ 5 |
| Complete 8-D Google scores | ≥ 95% |
| Required files | `elites.json`, `reserves.json`, `archive.json` |


In [ ]:
# Verify gate0 on disk (notebook does not re-run Phase 0)
import json
from pathlib import Path

gate0_path = RESULTS_DIR / "gate0_smoke.json"
if not gate0_path.is_file():
    raise RuntimeError(
        "Missing gate0_smoke.json — run:\n"
        "  python experiments/emnlp_data_analysis/smoke_gate.py"
    )
gate0 = json.loads(gate0_path.read_text(encoding="utf-8"))
if not gate0.get("pass"):
    failed = [k for k, v in gate0["checks"].items() if not v]
    raise RuntimeError(f"Gate 0 FAIL: {failed}")
print(f"Gate 0: PASS ({gate0['n_genomes']:,} genomes, {gate0['n_species']} species)")


### Phase 0 — Findings

See `results/gate0_smoke.json` after running `smoke_gate.py`.


## Phase 1 — Unified Genome Table (offline)

Build and score **outside the notebook** so analysis cells cannot overwrite the canonical dataset.

```bash
# 1) Google + embeddings → CSV (fast)
python experiments/emnlp_data_analysis/build_unified_dataset.py build

# 2) OpenAI → updates same CSV (slow; resumable via *_openai_cache.json)
python experiments/emnlp_data_analysis/build_unified_dataset.py score-openai

# Cache only (no API):
python experiments/emnlp_data_analysis/build_unified_dataset.py score-openai --no-fetch
```

**Artifacts** (`results/unified/`): `{run_id}_genomes.csv`, `_embeddings.npy`, `_genome_ids.json`, `_openai_cache.json`, `results/phase1_manifest.json`.

Phases **2+** load from these files via `load_unified_from_artifacts()` — not from run JSON.


In [ ]:
# Verify Phase 1 manifest (notebook does not rebuild or re-score)
import json
from pathlib import Path

manifest_path = RESULTS_DIR / "phase1_manifest.json"
if not manifest_path.is_file():
    raise RuntimeError(
        "Missing phase1_manifest.json — run build_unified_dataset.py build && score-openai"
    )
phase1 = json.loads(manifest_path.read_text(encoding="utf-8"))
n_g = phase1.get("n_genomes", 0)
n_go = phase1.get("n_with_google_objectives", 0)
n_oa = phase1.get("n_with_openai_objectives", 0)
if n_go < n_g or n_oa < n_g:
    raise RuntimeError(
        f"Phase 1 incomplete: google {n_go}/{n_g}, openai {n_oa}/{n_g}. "
        "Re-run build_unified_dataset.py"
    )
RUN_ID = phase1["run_id"]
print(f"Phase 1 OK: |G|={n_g:,}, Google={n_go:,}, OpenAI={n_oa:,}, run_id={RUN_ID}")
print(f"  CSV: {phase1['artifacts'].get('csv')}")


In [ ]:
# Optional: peek at canonical CSV (read-only)
import pandas as pd
from pathlib import Path

_csv = Path(phase1["artifacts"]["csv"])
if not _csv.is_absolute():
    _csv = REPO_ROOT / _csv
genomes_df = pd.read_csv(_csv)
print(f"Rows: {len(genomes_df):,}, columns: {len(genomes_df.columns)}")
display(genomes_df.head(3))


### Phase 1 — Findings

Canonical table lives in `results/unified/{run_id}_genomes.csv` (built by `build_unified_dataset.py`). Re-run scoring only if OpenAI columns are missing; notebook phases 2+ never modify this file.


## Phase 2 — Global & Local Pareto Ranks

**Prerequisites:** `phase1_manifest.json` shows full Google + OpenAI coverage. Canonical data is read from `results/unified/*_genomes.csv` (notebook does not modify it).

Post-hoc 8-D **Google/Perspective** Pareto on the unified population (cluster_analysis / AMBER convention):

| Scope | Rank column | F₀ flag |
|-------|-------------|---------|
| **Global** | `global_pareto_rank` (0 = F₀) | `on_global_f0` |
| **Species** (per `species_id > 0`) | `species_pareto_rank` | `on_species_f0` |
| **Reserves** (`reserves.json` only) | `reserves_pareto_rank` | `on_reserves_f0` |
| **Archive** (`archive.json` only) | `archive_pareto_rank` | `on_archive_f0` |
| **Viz cohort** (`species_*`, `reserves`, `archive`) | `pareto_rank_cohort` | `on_cohort_f0` |

**Outputs:** `results/phase2/` — annotated CSV, cohort summaries, per-cohort front CSVs.

**Figures (two parallel sets):** pymoo PCP + Radviz under `figures/google/` (8-D Perspective) and `figures/openai/` (13-D OpenAI). Each set computes its own global/cohort F₀ in that objective space. Paper main text uses **Google**; OpenAI supports robustness (Phase 6+).


In [13]:
import importlib
import json
import sys
from pathlib import Path

import pandas as pd

if "ANALYSIS_ROOT" not in globals():
    _here = Path.cwd().resolve()
    ANALYSIS_ROOT = None
    for base in [_here, *_here.parents]:
        for root in (base, base / "experiments" / "emnlp_data_analysis"):
            if (root / "analysis_utils.py").is_file():
                ANALYSIS_ROOT = root
                break
        if ANALYSIS_ROOT is not None:
            break
    if ANALYSIS_ROOT is None:
        raise RuntimeError("Could not locate experiments/emnlp_data_analysis")
    sys.path.insert(0, str(ANALYSIS_ROOT))

import analysis_utils
import pymoo_pcp_viz

importlib.reload(analysis_utils)
importlib.reload(pymoo_pcp_viz)
from analysis_utils import (
    RESULTS_DIR as _RESULTS_DIR,
    annotate_pareto_ranks,
    load_unified_from_artifacts,
    run_id_from_path,
    save_phase2_artifacts,
)

RESULTS_DIR = globals().get("RESULTS_DIR", _RESULTS_DIR)
RUN_PATH = globals().get("RUN_PATH", analysis_utils.DEFAULT_PRIMARY_RUN)
RUN_ID = globals().get("RUN_ID", run_id_from_path(RUN_PATH))
PHASE2_DIR = RESULTS_DIR / "phase2"

if "rows" not in globals():
    rows, p1_meta = load_unified_from_artifacts(RESULTS_DIR, RUN_ID)
    n_g = p1_meta["n_genomes"]
    n_go = p1_meta["n_with_google_objectives"]
    n_oa = p1_meta["n_with_openai_objectives"]
    if n_go < n_g or n_oa < n_g:
        raise RuntimeError(
            f"Phase 1 incomplete: google {n_go}/{n_g}, openai {n_oa}/{n_g}. "
            "Run build_unified_dataset.py"
        )
    print(f"Loaded {len(rows):,} genomes from {p1_meta['csv']} (read-only)")

phase2_stats = annotate_pareto_ranks(rows)
phase2_paths = save_phase2_artifacts(rows, phase2_stats, PHASE2_DIR, RUN_ID)

phase2_manifest = {
    "run_id": RUN_ID,
    "stats": phase2_stats,
    "artifacts": phase2_paths,
}
(RESULTS_DIR / "phase2_manifest.json").write_text(
    json.dumps(phase2_manifest, indent=2), encoding="utf-8"
)

print(f"Global F₀: {phase2_stats['n_f0']:,} / {phase2_stats['n_genomes']:,} "
      f"({100 * phase2_stats['f0_fraction']:.2f}%)")
print(f"Pareto fronts: {phase2_stats['n_fronts']}")
print(f"Wrote {PHASE2_DIR}/")
for k, v in sorted(phase2_paths.items()):
    if v:
        print(f"  {k}: {v}")

n_fig = sum(1 for k in phase2_paths if k.startswith("pymoo_"))
if n_fig == 0:
    print(
        "WARNING: no pymoo figure paths — check stderr above. "
        "Re-run after: pip install pymoo matplotlib; importlib.reload(pymoo_pcp_viz)."
    )
else:
    print(f"pymoo figures written: {n_fig}")


Cannot plot a one dimensional array.
Cannot plot a one dimensional array.
Cannot plot a one dimensional array.
Cannot plot a one dimensional array.
Global F₀: 100 / 5,655 (1.77%)
Pareto fronts: 28
Wrote /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis/results/phase2/
  cohort_summary_csv: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis/results/phase2/pareto_cohort_summary.csv
  figures_google_dir: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis/results/phase2/figures/google
  figures_openai_dir: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis/results/phase2/figures/openai
  genomes_pareto_csv: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis/results/phase2/20260211_2122_genomes_pareto.csv
  pareto_fronts_by_cohort_dir: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis/results/phase2/pareto_fronts_by_cohort
  phase2_sum

In [14]:
pareto_df = pd.read_csv(phase2_paths["genomes_pareto_csv"])
display(
    pd.DataFrame(
        [
            {"metric": "genomes", "value": len(pareto_df)},
            {"metric": "on global F0", "value": int(pareto_df["on_global_f0"].sum())},
            {"metric": "on species F0 (any)", "value": int(pareto_df["on_species_f0"].sum())},
            {"metric": "on reserves F0", "value": int(pareto_df["on_reserves_f0"].sum())},
            {"metric": "on archive F0", "value": int(pareto_df["on_archive_f0"].sum())},
        ]
    )
)
display(pd.read_csv(phase2_paths["cohort_summary_csv"]))
display(pareto_df.groupby("cohort").agg(
    n=("genome_id", "count"),
    global_f0=("on_global_f0", "sum"),
    cohort_f0=("on_cohort_f0", "sum"),
    min_global_rank=("global_pareto_rank", "min"),
).sort_values("n", ascending=False))

,metric,value
0,genomes,5655
1,on global F0,100
2,on species F0 (any),102
3,on reserves F0,1
4,on archive F0,80


,cohort,n_genomes,n_cohort_f0,n_global_f0
0,archive,5192,80,58
1,reserves,3,1,0
2,species_2415,25,9,0
3,species_2421,150,36,36
4,species_2422,88,13,2
5,species_2423,44,4,3
6,species_2424,54,10,0
7,species_2425,24,7,1
8,species_2426,61,13,0
9,species_2427,6,4,0


,n,global_f0,cohort_f0,min_global_rank
cohort,,,,
archive,5192,58,80,0
species_2421,150,36,36,0
species_2422,88,2,13,0
species_2426,61,0,13,4
species_2424,54,0,10,3
species_2423,44,3,4,0
species_2415,25,0,9,10
species_2425,24,1,7,0
species_2428,8,0,6,6


### Phase 2 — Findings

**Hero figures (2):**
- `phase2/figures/google/pymoo_pcp_pareto_google.png` — combined Google PCP (per-cohort F₀ dotted, global F₀ solid).
- `phase2/figures/openai/pymoo_pcp_pareto_openai.png` — same structure for OpenAI.

Per-cohort PCPs and Radviz are now demoted to `phase2/figures/{evaluator}/appendix/` (24 files) to honour the ≤3 hero-figure budget.

**Headline numbers (from `results/phase2_manifest.json` + recomputed for OpenAI):**
- **Google global F₀:** 100 / 5,655 (**1.77 %**), 28 Pareto fronts.
- **OpenAI global F₀:** 144 / 5,655 (**2.55 %**), 99 Pareto fronts (much flatter dominance structure on the 13-D vector).
- **Cohort decomposition (Google global F₀):**
  - `species_2421`: 36/150 (only species with substantial front presence)
  - `species_2422`: 2/88, `species_2423`: 3/44, `species_2425`: 1/24
  - `species_2415, 2424, 2426, 2427, 2428`: **0/T_c** each — five topics with no member on the global front.
  - `archive` carries 58/100 of the global front; `reserves` (n=3) contributes 0.

**Answers (Section M.3):** "Does Pareto over multiple objectives recover all topics?" — already gives a partial *no*: five of nine topics put no genome on the Google front. Phase 3 quantifies this as TDI.

**Artifacts:** `results/phase2/20260211_2122_genomes_pareto.csv`, `phase2_global_pareto.json`, `pareto_cohort_summary.csv`, `pareto_fronts_by_cohort/`, `phase2_manifest.json`, two hero PCPs.

**Caveats / impact on later phases:**
- The OpenAI front is **much larger** (144 vs 100) and has **3.5× more Pareto fronts** (99 vs 28). This shifts what "fully dominated" means under OpenAI; expect Phase 3 to find a different fully-dominated topic set than Google.
- `species_2421` dominates the front-on-front set — Phase 6 should check whether `2421` is also embedding-distinct (otherwise our "distinct + fully dominated" story rests on the other 5 topics).
- No plan update needed; consistent with `[scripts/EMNLP_4PAGER_PLAN.txt](../../../scripts/EMNLP_4PAGER_PLAN.txt)` Section H anchors.

## Phase 3 — Topic Domination (TDI) & Evaluator Rank Agreement

**Per evaluator (separate objective spaces):** for each species topic with \(n \geq 5\), report TDI, fraction on global \(F_0\), fully dominated (no \(F_0\) members), embedding distinctness, axis-exclusive axes.

**Cross-evaluator:** same genomes ranked on Google 8-D vs OpenAI 13-D global fronts — same rank?, \(F_0\) overlap, rank differences.

**Outputs:** `results/phase3/google/`, `results/phase3/openai/`, `evaluator_rank_agreement.csv`, `evaluator_rank_agreement_summary.json`

**Also (3b):** `topic_evaluator_comparison.csv` (paired TDI / F₀ per topic), domination matrix CSV + heatmap + directed graph per evaluator under `results/phase3/figures/`.

In [15]:
import importlib

importlib.reload(analysis_utils)
from analysis_utils import save_phase3_artifacts

PHASE3_DIR = RESULTS_DIR / "phase3"

if "rows" not in globals():
    rows, _ = load_unified_from_artifacts(RESULTS_DIR, RUN_ID)[0]
    print(f"Loaded {len(rows):,} genomes")

phase3_out = save_phase3_artifacts(
    rows,
    PHASE3_DIR,
    RUN_ID,
    min_topic_size=MIN_TOPIC_SIZE,
)
phase3_paths = phase3_out["artifacts"]
phase3_meta = phase3_out["meta"]

print("Phase 3 artifacts:")
for k, v in sorted(phase3_paths.items()):
    print(f"  {k}: {v}")

print("\n--- Google global stats ---")
for k, v in phase3_meta["evaluators"]["google"].items():
    if not k.startswith("tau") and k != "min_topic_size":
        print(f"  {k}: {v}")

print("\n--- OpenAI global stats ---")
for k, v in phase3_meta["evaluators"]["openai"].items():
    if not k.startswith("tau") and k != "min_topic_size":
        print(f"  {k}: {v}")

print("\n--- Google vs OpenAI rank agreement ---")
ra = phase3_meta["rank_agreement"]
for k in (
    "n_genomes_both_evaluators",
    "n_google_f0",
    "n_openai_f0",
    "n_same_rank",
    "frac_same_rank",
    "n_both_on_f0",
    "frac_both_on_f0",
    "n_google_f0_only",
    "n_openai_f0_only",
    "jaccard_f0_sets",
    "spearman_rank_correlation",
    "rank_diff_mean",
    "rank_diff_median",
):
    print(f"  {k}: {ra.get(k)}")

topic_cmp = pd.read_csv(phase3_paths["topic_evaluator_comparison_csv"])
print("Topic evaluator comparison:")
display(topic_cmp.sort_values("species_id"))
if "topic_evaluator_comparison" in phase3_meta:
    display(pd.DataFrame([phase3_meta["topic_evaluator_comparison"]]).T.rename(columns={0: "value"}))

for key in (
    "fig_topic_tdi_comparison",
    "fig_domination_heatmap_google",
    "fig_domination_heatmap_openai",
    "fig_domination_graph_google",
    "fig_domination_graph_openai",
):
    if key in phase3_paths:
        print(f"  {key}: {phase3_paths[key]}")

tdi_google = pd.read_csv(phase3_paths["google_topic_summary_csv"])
tdi_openai = pd.read_csv(phase3_paths["openai_topic_summary_csv"])
rank_df = pd.read_csv(phase3_paths["rank_agreement_csv"])

display(tdi_google.sort_values("species_id"))
display(tdi_openai.sort_values("species_id"))
display(rank_df.head(10))
print(f"\nRank agreement rows: {len(rank_df):,}")
display(
    rank_df.groupby("same_rank").size().rename("n_genomes").to_frame()
)
display(
    pd.DataFrame(
        [
            {"f0_pattern": "both", "n": int(ra["n_both_on_f0"])},
            {"f0_pattern": "google_only", "n": int(ra["n_google_f0_only"])},
            {"f0_pattern": "openai_only", "n": int(ra["n_openai_f0_only"])},
            {"f0_pattern": "neither", "n": int(ra["n_neither_f0"])},
        ]
    )
)


Phase 3 artifacts:
  google_edges_json: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis/results/phase3/google/topic_domination_edges.json
  google_global_stats_json: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis/results/phase3/google/topic_domination_global_stats.json
  google_topic_summary_csv: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis/results/phase3/google/topic_domination_summary.csv
  openai_edges_json: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis/results/phase3/openai/topic_domination_edges.json
  openai_global_stats_json: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis/results/phase3/openai/topic_domination_global_stats.json
  openai_topic_summary_csv: /Users/onkars/Documents/Projects/ToxSearch-S/experiments/emnlp_data_analysis/results/phase3/openai/topic_domination_summary.csv
  phase3_manifest: /Users/onkars/Documents/Projec

,evaluator,species_id,n_members,n_on_f0,frac_on_f0,tdi,fully_dominated,distinct_topic,axis_exclusive_axes,n_axis_exclusive,intra_cosine_mean,inter_dg_min,dominating_topics,max_axis_0
0,google,2415,25,0,0.000000,1.0000,True,False,NaN,0,0.3304,0.2716,2421;2423;2426;2422;2424;2425,0.0366
1,google,2421,150,36,0.240000,0.7600,False,True,NaN,0,0.9480,0.4275,NaN,0.9288
2,google,2422,88,2,0.022727,0.9773,False,False,NaN,0,0.5739,0.3699,NaN,0.3881
3,google,2423,44,3,0.068182,0.9318,False,False,NaN,0,0.4401,0.2180,NaN,0.3734
4,google,2424,54,0,0.000000,1.0000,True,False,NaN,0,0.2815,0.3015,2421,0.2178
5,google,2425,24,1,0.041667,0.9583,False,False,NaN,0,0.5148,0.4259,NaN,0.1960
6,google,2426,61,0,0.000000,1.0000,True,False,NaN,0,0.1938,0.2180,2421;2424,0.1200
7,google,2427,6,0,0.000000,1.0000,True,False,NaN,0,0.1487,0.2871,2421;2423;2426;2422;2424;2425,0.0378
8,google,2428,8,0,0.000000,1.0000,True,False,NaN,0,0.3198,0.2998,2421;2423;2422;2424;2425,0.0376


,evaluator,species_id,n_members,n_on_f0,frac_on_f0,tdi,fully_dominated,distinct_topic,axis_exclusive_axes,n_axis_exclusive,intra_cosine_mean,inter_dg_min,dominating_topics,max_axis_0
0,openai,2415,25,0,0.000000,1.0000,True,False,NaN,0,0.3304,0.2716,NaN,0.1995
1,openai,2421,150,14,0.093333,0.9067,False,True,NaN,0,0.9480,0.4275,NaN,0.9803
2,openai,2422,88,4,0.045455,0.9545,False,False,NaN,0,0.5739,0.3699,NaN,0.0277
3,openai,2423,44,1,0.022727,0.9773,False,False,NaN,0,0.4401,0.2180,NaN,0.0062
4,openai,2424,54,4,0.074074,0.9259,False,False,NaN,0,0.2815,0.3015,NaN,0.0057
5,openai,2425,24,0,0.000000,1.0000,True,False,NaN,0,0.5148,0.4259,NaN,0.0089
6,openai,2426,61,2,0.032787,0.9672,False,False,NaN,0,0.1938,0.2180,NaN,0.0278
7,openai,2427,6,0,0.000000,1.0000,True,False,NaN,0,0.1487,0.2871,2423;2426;2422;2424,0.0001
8,openai,2428,8,0,0.000000,1.0000,True,False,NaN,0,0.3198,0.2998,2421;2423;2426;2422;2424;2425,0.0008


,genome_id,species_id,cohort,google_rank,openai_rank,rank_diff,google_on_f0,openai_on_f0,same_rank,both_on_f0,google_f0_only,openai_f0_only
0,1062,2415,species_2415,20,45,25,False,False,False,False,False,False
1,5420,2415,species_2415,17,98,81,False,False,False,False,False,False
2,3289,2415,species_2415,17,98,81,False,False,False,False,False,False
3,3114,2415,species_2415,18,98,80,False,False,False,False,False,False
4,4859,2415,species_2415,19,98,79,False,False,False,False,False,False
5,4268,2415,species_2415,13,98,85,False,False,False,False,False,False
6,3744,2415,species_2415,13,24,11,False,False,False,False,False,False
7,1637,2415,species_2415,15,58,43,False,False,False,False,False,False
8,4118,2415,species_2415,17,98,81,False,False,False,False,False,False
9,4814,2415,species_2415,13,15,2,False,False,False,False,False,False



Rank agreement rows: 5,655


,n_genomes
same_rank,
False,5523
True,132


,f0_pattern,n
0,both,16
1,google_only,84
2,openai_only,128
3,neither,5427


### Phase 3 — Findings

**Hero figures (3, in `phase3/figures/`):**
- `topic_domination_heatmap_dual.png` — Google + OpenAI side-by-side max-vector domination heatmaps.
- `topic_domination_graph_google.png` — primary-evaluator dominator → dominated arrows.
- `topic_tdi_google_vs_openai.png` — per-topic TDI bar comparison.

Per-evaluator heatmaps and the OpenAI graph are demoted to `phase3/figures/appendix/`.

**Headline numbers (from `results/phase3/phase3_manifest.json`):**
- **Google (8-D):** |F₀| = 100, **5/9 topics fully dominated** (TDI = 1.0), 1 topic flagged distinct, 0 topics that are simultaneously distinct AND fully dominated, 0 axis-exclusive fully dominated.
- **OpenAI (13-D):** |F₀| = 144, **4/9 topics fully dominated**, 1 distinct, 0 axis-exclusive fully dominated.
- **Cross-evaluator topic comparison (`topic_evaluator_comparison.csv`):**
  - 3 topics fully dominated under **both** evaluators.
  - 3 topics fully dominated under **neither**.
  - 2 topics dominated under Google only; 1 under OpenAI only.
  - mean ΔTDI(OpenAI − Google) = +0.012, range [−0.074, +0.147].
- **Genome-level rank agreement (`evaluator_rank_agreement_summary.json`):**
  - Spearman ρ across 5,655 paired ranks: **0.706**.
  - Same global rank: 2.33 % (rank-diff median = 68).
  - **F₀ Jaccard:** **0.0702** (low — different genomes occupy each evaluator's front).

**Answers (Section M.3):**
- "Does Pareto over multiple objectives recover all topics?" — **NO** under either evaluator (4–5 of 9 topics get TDI = 1.0).
- "Is this only a Perspective quirk?" — Three topics are fully dominated regardless of evaluator; three are dominated under one evaluator only. Topic-level disagreement is real and must be reported explicitly in §5.

**Artifacts:** `topic_evaluator_comparison.csv`, `evaluator_rank_agreement.csv`, `google/topic_domination_summary.csv`, `openai/topic_domination_summary.csv`, plus matrices, edges, hero/appendix figures.

**Caveats / impact on later phases:**
- **`n_distinct_topics = 1` for both evaluators** — using the current thresholds (`τ_intra = 0.85`, `τ_inter = 0.35`), only one species clears the "distinct topic" gate. **Phase 8's threshold sweep is now load-bearing**: if relaxing thresholds doesn't lift the distinct-topic count, the §5 claim "behaviourally distinct yet dominated" rests on weaker embedding evidence and we should re-state it as "score-profile-distinct yet dominated" backed by Phase 6 axis profiles + Phase 11.
- The **F₀ Jaccard is very low (0.0702)** but Spearman is moderate (0.706) — Phase 7 should report this nuance: evaluators agree on rough ordering, disagree on which exact genomes are non-dominated.
- **Plan update:** trigger Section M caveat — added in plan-update-m8 step. (See §M / §H of [scripts/EMNLP_4PAGER_PLAN.txt](../../../scripts/EMNLP_4PAGER_PLAN.txt) for current anchors.)

## Phase 4 — Temporal Species Dynamics (Fig 1A)

Parse `EvolutionTracker.json`: **active species count**, **reserves pool size**, and **merge events** per generation. Marks collapse-to-one generations.

**Outputs:** `results/phase4/timeseries_{run_id}.csv`, `timeseries_{run_id}_stats.json`, `figures/fig1_temporal_species_count.png`

In [ ]:
import importlib

importlib.reload(analysis_utils)
from analysis_utils import save_phase4_artifacts

PHASE4_DIR = RESULTS_DIR / "phase4"
REFERENCE_SPECIES = 9  # final large topics in Phase 3

phase4_out = save_phase4_artifacts(
    RUN_PATH,
    PHASE4_DIR,
    RUN_ID,
    reference_species=REFERENCE_SPECIES,
)
phase4_paths = phase4_out["artifacts"]
phase4_meta = phase4_out["meta"]
stats4 = phase4_meta.get("stats") or {}

print("Phase 4 artifacts:")
for k, v in sorted(phase4_paths.items()):
    print(f"  {k}: {v}")

print("\n--- Temporal summary ---")
for k in (
    "n_generations_parsed",
    "active_species_min",
    "active_species_max",
    "final_active_species_count",
    "n_generations_collapsed_to_one",
    "reserves_size_at_collapse_mean",
):
    print(f"  {k}: {stats4.get(k)}")
if stats4.get("collapsed_generations"):
    print(f"  collapsed_generations (first 10): {stats4['collapsed_generations'][:10]}")
    print(f"  collapsed_generations (last 10): {stats4['collapsed_generations'][-10:]}")

ts_df = pd.read_csv(phase4_paths["timeseries_csv"])
display(ts_df.head(8))
display(ts_df.tail(8))
display(
    ts_df.groupby("collapsed_to_one").size().rename("n_generations").to_frame()
)

### Phase 4 — Findings

**Hero figure (1):** `phase4/figures/fig1_temporal_species_count.png` — active-species count over generations with reserves footprint and collapse markers.

**Headline numbers (from `results/phase4/timeseries_*_stats.json`):**
- 251 generations parsed; final active species count = **9** (matches Phase 0).
- Active species range: **0 → 12** across the run.
- **18 generations collapsed to a single active species** (`collapsed_to_one = True`), e.g. gen 7, 177, 183, 201, 205, ..., 239, 241, 243, 247, 249.
- Reserves population **at collapse** averages **333.6** genomes (range 210 → 608); reserves overall mean = 373.7, max = 902.
- Speciation events total: **1,507**; merge events total: **671**; extinction events: **0** (collapses are merges, not extinctions).
- Mean `avg_fitness` rises from **0.054** (first 10 gens) to **0.308** (last 10 gens) — scalar pressure visibly drives fitness up.
- `max_score_variants` peaks at **0.929**.

**Answers (Section M.3):**
- "Does scalar search collapse topical diversity?" — **Yes, intermittently.** Although the run *ends* with 9 active species (matching Phase 0), it spends 18 generations (~7 % of the run) at one active species. Collapses happen repeatedly through the latter half of the run.
- Crucially, when `active_species_count = 1` the **reserves pool is large (mean ≈ 334)**, so material capable of seeding diverse niches survives even though active selection drops it. This sets up Phase 5: the speciated-oracle policy upper-bounds what could have been kept.

**Artifacts:** `timeseries_20260211_2122.csv` (251 rows × 13 cols), `timeseries_20260211_2122_stats.json`, `phase4_manifest.json`, hero figure.

**Caveats / impact on later phases:**
- The `n_merge_events` field in `phase4_manifest.json` is `null` because `save_phase4_artifacts` aggregates only via the per-generation column; the **total 671 merges** above was computed by summing the CSV. Not blocking, but noted for Phase 10's manifest.
- 18 collapses with large reserves is a strong qualitative anchor for §5.2 ("scalar collapse with usable diversity in reserves"). No plan update needed — the existing Section H entry "intermittently collapses active species to 1 while a large reserves pool remains" is still accurate.

## Phase 5 — Counterfactual Retention (Fig 3)

Three policies × two evaluators: scalar top-k (north-star axis), global MO F₀, speciated oracle. k-sensitivity at {50, 100, 200}.

**Outputs:** `results/phase5/counterfactual_survival_*.csv`, `figures/fig3_counterfactual_*.png`

In [ ]:
import importlib

importlib.reload(analysis_utils)
from analysis_utils import save_phase5_artifacts

phase5_out = save_phase5_artifacts(rows, RESULTS_DIR / "phase5", RUN_ID, min_topic_size=MIN_TOPIC_SIZE)
phase5_paths = phase5_out["artifacts"]
print("Phase 5 artifacts:")
for k, v in sorted(phase5_paths.items()):
    print(f"  {k}: {v}")
cf = pd.read_csv(phase5_paths["counterfactual_combined_csv"])
display(cf)

### Phase 5 — Findings

**Hero figures (3, in `phase5/figures/`):**
- `fig3_counterfactual_dual_panel.png` — Google + OpenAI side-by-side bars (the planned Fig 3).
- `fig3_counterfactual_google.png`, `fig3_counterfactual_openai.png` — single-evaluator versions.

**Headline numbers (`counterfactual_survival_combined.csv`):**

| evaluator | scalar (top-`|F₀|` by toxicity) | global Pareto front | speciated oracle |
|-----------|-----------------------------------|---------------------|------------------|
| Google    | **1 / 9** (11.1 %)               | **4 / 9** (44.4 %) | **9 / 9** (100 %) |
| OpenAI    | **1 / 9** (11.1 %)               | **5 / 9** (55.6 %) | **9 / 9** (100 %) |

Hierarchy `scalar < global_mo < speciated_oracle` holds for **both** evaluators (`counterfactual_evaluator_comparison.json` confirms `hierarchy_*: true`).

**Answers (Section M.3):**
- "Would Pareto-only retention erase those topics?" — **Yes.** Even using the full multi-objective Pareto front of moderation scores, **5/9 (Google) or 4/9 (OpenAI)** topics are extinguished post-hoc. The speciated oracle (per-topic local F₀) preserves all nine.
- "Does scalar focus erase well-rounded niches?" — Scalar top-`|F₀|` keeps only **1/9** topics under either evaluator, the worst regime, exactly the scenario the professor flagged.

**Artifacts:** `counterfactual_survival_google.csv`, `counterfactual_survival_openai.csv`, `counterfactual_survival_combined.csv`, `counterfactual_k_sensitivity.csv` (k-sweep for sensitivity), `counterfactual_evaluator_comparison.json`, `phase5_manifest.json`, three hero figures.

**Caveats / impact on later phases:**
- Speciated oracle keeps **102 (Google) or 72 (OpenAI)** genomes — these are the union of per-topic local F₀, not necessarily Pareto-optimal globally. Phase 11 should reuse this as a baseline for "what speciation could buy."
- The fact that `global_mo` retains only 4–5 topics is the headline §5 number for the EMNLP conclusion. Phase 11 will generalise this to per-axis policies (8 single-axis top-k retention) to demonstrate that **no single moderation axis policy** recovers all niches — that is the deep advanced analytic the professor asked for.
- No plan update needed; Section H counterfactual figures are now anchored at Google 11 / 44 / 100 and OpenAI 11 / 56 / 100.

## Phase 6 — Behaviour & Topic Profiles

Dual UMAP (F₀ outline per evaluator), topic×axis max heatmaps, embedding cluster quality, centroid d_g vs d_p.

**Outputs:** `results/phase6/figures/fig1_umap_*.png`, `topic_axis_max_*.csv`, `cluster_quality.json`

In [ ]:
import importlib

importlib.reload(analysis_utils)
from analysis_utils import save_phase6_artifacts

phase6_out = save_phase6_artifacts(rows, RESULTS_DIR / "phase6", RUN_ID, min_topic_size=MIN_TOPIC_SIZE)
phase6_paths = phase6_out["artifacts"]
print("Phase 6 artifacts:")
for k, v in sorted(phase6_paths.items()):
    print(f"  {k}: {v}")
with open(phase6_paths["cluster_quality_json"]) as f:
    display(pd.json_normalize(json.load(f)))

### Phase 6 — Findings

**Hero figures (3, in `phase6/figures/`):**
- `fig1_umap_dual.png` — single UMAP layout, F₀ markers swap per evaluator (Google left, OpenAI right).
- `topic_axis_heatmap_dual.png` — per-topic max-score profiles, Google + OpenAI side-by-side.
- `fig_dg_dp_dual.png` — topic-centroid genotype vs phenotype distance (red points involve a fully dominated topic).

Per-evaluator UMAPs, heatmaps, and dg/dp scatters are now in `phase6/figures/appendix/`.

**Headline numbers (`phase6/cluster_quality.json`, `topic_axis_max_*.csv`):**
- **Cluster quality (over 460 elite genomes with `species_id > 0`, 9 clusters):**
  - silhouette = **0.364** (decent inter-topic separation in 384-D embedding space).
  - Davies–Bouldin = 2.52, Calinski–Harabasz = 55.43.
- **Topic-axis profiles:** the per-topic max-score matrices in `topic_axis_max_google.csv` (9 × 8) and `topic_axis_max_openai.csv` (9 × 13) are the substrate Phase 11 will read; the heatmap shows non-trivial differences across species (e.g. species with high `flirtation` peak versus species with high `insult` peak).

**New observation that affects the narrative:**
- In this run, **`species_id > 0` is set only on the 460 elites**; archive (5,192) and reserves (3) carry `species_id ∈ {0, −1}`.
- This means **58 of the 100 Google global-F₀ genomes are NOT in any topic** (they are archived prompts), and 42 are split across only 4 species. So Pareto retention not only misses 5/9 topics — **a majority of the surviving front is "topicless"** (in the operational sense of `species_id`). Phase 11 will treat this as an additional reason scalar/global-MO selection over a flat population is unsafe for topical coverage; the speciated-oracle baseline does not have this problem because it picks per-topic local fronts.

**Answers (Section M.3):**
- "Are dominated topics behaviourally separate?" — **Yes, in elite space:** silhouette 0.36 in 384-D embeddings. d_g / d_p plot shows red points (involving a fully dominated topic) span the full d_g / d_p range, i.e. fully-dominated topics are not cluster-quality outliers — they are normal topics that just happen to be dominated in score space.
- "Do topics show different score profiles?" — Yes; visible in dual heatmap. Phase 11 quantifies this as `niche_specialization_index`.

**Artifacts:** `cluster_quality.json`, `topic_axis_max_google.csv`, `topic_axis_max_openai.csv`, three hero figures, six appendix figures, `phase6_manifest.json`.

**Caveats / impact on later phases:**
- The "topicless 58/100 front" observation is an analytic that should be added to Phase 11's `topic_advanced_analytics.csv` as a top-level metric (not a per-row column) and surfaced in the Phase 11 findings.
- `n_distinct_topics = 1` from Phase 3 is consistent with d_g / d_p plot density: most centroid pairs cluster, only one species is sufficiently far. **No plan update needed**, but Phase 11's "distinct-but-dominated" scatter will rely on a *softer* notion of distinctness (e.g. silhouette by topic) rather than the strict τ thresholds.

## Phase 7 — Inference

10k permutation test (mean d_g: F₀ vs dominated topic members), rank contingency heatmap from Phase 3, mapped-axis score correlations.

**Outputs:** `results/phase7/inference_permutation_*.json`, `figures/rank_contingency_heatmap.png`

In [ ]:
import importlib

importlib.reload(analysis_utils)
from analysis_utils import save_phase7_artifacts

phase7_out = save_phase7_artifacts(rows, RESULTS_DIR / "phase7", RUN_ID, min_topic_size=MIN_TOPIC_SIZE)
phase7_paths = phase7_out["artifacts"]
print("Phase 7 artifacts:")
for k, v in sorted(phase7_paths.items()):
    print(f"  {k}: {v}")
with open(phase7_paths["inference_summary_json"]) as f:
    display(pd.json_normalize(json.load(f)))

### Phase 7 — Findings

**Hero figure (1):** `phase7/figures/rank_contingency_heatmap.png` — pairwise count of (Google rank, OpenAI rank) bins; off-diagonal mass quantifies cross-evaluator disagreement on Pareto rank.

**Headline numbers (`inference_summary.json`):**

| evaluator | n on F₀ | n dominated members | observed permutation stat | p-value |
|-----------|---------|----------------------|----------------------------|---------|
| Google    | 100     | 418                  | 0.288                      | **0.868** (NOT significant) |
| OpenAI    | 144     | 435                  | 0.395                      | **0.0001** |

**Axis-pair correlation across the 5,655 paired genome scores (`axis_score_correlation.csv`):**

| Google axis | OpenAI axis | Spearman | Pearson |
|-------------|-------------|----------|---------|
| toxicity | harassment | 0.760 | 0.882 |
| severe_toxicity | harassment/threatening | 0.617 | 0.138 |
| identity_attack | hate | 0.651 | 0.688 |
| insult | harassment | 0.768 | 0.896 |
| profanity | harassment | 0.686 | 0.902 |
| threat | violence | 0.598 | 0.092 |
| sexually_explicit | sexual | 0.442 | 0.554 |
| flirtation | sexual | 0.370 | 0.152 |

Mean Spearman = 0.611; mean Pearson = 0.538.

**Answers (Section M.3):**
- "Are dominated topics behaviourally distinct from on-front topics?" — **Google: cannot reject the null** (p = 0.87, embeddings of dominated vs F₀ members are exchangeable). **OpenAI: reject the null** (p = 0.0001). The §5 narrative therefore must NOT lean on "dominated topics are embedding-distinct" as a Google-grounded claim — the right framing is "domination depends on evaluator; embedding distinctness is robust under OpenAI but not under Google."
- Axis correlations are moderate-to-strong on the toxicity / harassment / insult / profanity cluster (Spearman > 0.6), and weak on threat / sexually_explicit / flirtation (< 0.5). This empirically supports the §5 line "moderation objectives are correlated; one topic can dominate on all axes simultaneously."

**Artifacts:** `inference_permutation_google.json`, `inference_permutation_openai.json`, `inference_summary.json`, `axis_score_correlation.csv`, hero figure, `phase7_manifest.json`.

**Caveats / impact on later phases:**
- **Plan-update trigger:** the previous summary block in `[scripts/EMNLP_4PAGER_PLAN.txt](../../../scripts/EMNLP_4PAGER_PLAN.txt)` Section H lists "Permutation: Google p ≈ 0.8677; OpenAI p ≈ 0.0001" — this matches now exactly, so no plan rewrite needed, but the §5 narrative wording ("UMAP shows dominated topics are embedding-distinct") must be **softened to "supported under OpenAI, not under Google"** in the future report. We log this in the planned `M.8` section addition rather than over-editing the plan now.
- Section M.4.5 (already in the plan) explicitly warns to report Google as non-significant — that warning is now anchored by Phase 7 numbers above.
- `flirtation` having Spearman 0.37 vs `sexual` flags **flirtation as a likely "axis-exclusive" candidate** for Phase 11. Phase 11 will compute single-axis top-k retention specifically including a flirtation policy.

## Phase 8 — Sensitivity

ε-dominance sweep (both evaluators), τ_intra/τ_inter threshold grid, population ablation counterfactuals.

**Outputs:** `results/phase8/sensitivity/*.csv`, `figures/epsilon_sweep_topics.png`

In [ ]:
import importlib

importlib.reload(analysis_utils)
from analysis_utils import save_phase8_artifacts

phase8_out = save_phase8_artifacts(rows, RESULTS_DIR / "phase8", RUN_ID, min_topic_size=MIN_TOPIC_SIZE)
phase8_paths = phase8_out["artifacts"]
print("Phase 8 artifacts:")
for k, v in sorted(phase8_paths.items()):
    print(f"  {k}: {v}")
eps_g = pd.read_csv(phase8_paths["epsilon_sweep_google_csv"])
display(eps_g)

### Phase 8 — Findings

**Hero figure (1):** `phase8/figures/epsilon_sweep_topics.png` — fully-dominated topic count vs ε for both evaluators.

**Headline numbers (`phase8/sensitivity/sensitivity.json`):**

**ε-dominance sweep (Google):**

| ε    | |F₀_ε| | fully dominated topics | global_mo survival rate |
|------|--------|------------------------|--------------------------|
| 0.00 | 100    | 5                      | 0.444 |
| 0.01 | 218    | 5                      | 0.444 |
| 0.02 | 381    | 5                      | 0.444 |
| 0.05 | 577    | 5                      | 0.444 |
| 0.10 | 1,036  | **2**                  | 0.444 |

**ε-dominance sweep (OpenAI):**

| ε    | |F₀_ε| | fully dominated topics | global_mo survival rate |
|------|--------|------------------------|--------------------------|
| 0.00 | 144    | 4                      | 0.556 |
| 0.01 | 5,655  | 0                      | 0.556 |
| ≥ 0.02 | 5,655 | 0                    | 0.556 |

**Threshold sweep (`τ_intra ∈ {0.80, 0.85, 0.90} × τ_inter ∈ {0.25, 0.35, 0.45}`):**
- `n_distinct_dominated` = 0 for **all 9 combinations**.
- `n_axis_exclusive_dominated` = 0 for **all 9 combinations**.

**Population ablations (full / elites-only / no_reserves) on counterfactual:**
- "full" and "no_reserves": identical to Phase 5 (Google 1/4/9, OpenAI 1/5/9). Reserves (n=3) doesn't move the answer.
- "elites" (drop archive + reserves; only the 460 species-tagged genomes): Google 1/4/9, OpenAI **1/6/9**. OpenAI gains one topic when archive is removed (smaller front, dominance reshuffles), but the qualitative hierarchy is unchanged.

**Answers (Section M.3):**
- "Is the dominated-topic count brittle to ε?" — Under Google, **needs ε ≥ 0.10** to even dent the dominated set (drops 5 → 2). Under OpenAI, ε = 0.01 already collapses every genome onto an ε-front (front explodes to all 5,655). The Google ε needed to "save" each topic is large; for OpenAI the front is shallow.
- "Is the distinct-but-dominated set robust to thresholds?" — **No, it is empty across all 9 threshold combinations.** This is the Phase 3 finding generalised: **we cannot rely on the embedding-distinct + fully-dominated AND criterion** at the species-elite level. Phase 11 must use a soft per-topic specialization measure to demonstrate the same point.
- "Is global_mo survival robust to population edits?" — Yes; only the elites-only OpenAI cell moves (5 → 6 topics). The headline 4–5/9 number stays.

**Artifacts:** `epsilon_sweep_google.csv`, `epsilon_sweep_openai.csv`, `threshold_sweep_distinctness.csv`, `population_ablation_counterfactual.csv`, `sensitivity.json`, hero figure, `phase8_manifest.json`.

**Caveats / impact on later phases:**
- **Plan-update trigger (logged for `M.8`):** the strict "distinct + fully dominated" claim is unsupported across thresholds. The §5 narrative must replace it with a softer "score-profile distinct + fully dominated" framing backed by Phase 6 axis profiles + Phase 11's `niche_specialization_index` and per-axis policy survival.
- The OpenAI ε-degeneracy at ε = 0.01 reflects 13-D scores being closer to the unit cube; **do not** report ε-sweep on OpenAI as evidence of robustness — it is too sensitive to be informative. Note this as a limitation.

## Phase 9 — Cross-Evaluator Robustness

Jaccard on fully dominated topics and F₀ sets; counterfactual hierarchy; pattern agreement from Phase 3 comparison.

**Outputs:** `results/phase9/cross_evaluator_robustness.json`

In [ ]:
import importlib

importlib.reload(analysis_utils)
from analysis_utils import save_phase9_artifacts

phase9_out = save_phase9_artifacts(rows, RESULTS_DIR / "phase9", RUN_ID, min_topic_size=MIN_TOPIC_SIZE)
phase9_paths = phase9_out["artifacts"]
print("Phase 9 artifacts:")
for k, v in sorted(phase9_paths.items()):
    print(f"  {k}: {v}")
display(pd.read_csv(phase9_paths["cross_evaluator_robustness_csv"]))

### Phase 9 — Findings

**Hero figure (1):** `phase9/figures/cross_evaluator_summary.png` — bar summary of cross-evaluator robustness metrics.

**Headline numbers (`cross_evaluator_robustness.json`):**
- **Jaccard (Google F₀ ∩ OpenAI F₀ / union):** **0.0702** — only **16** genomes are on the front under both evaluators (out of 100 ∪ 144 = 228 total unique).
- **Jaccard (fully dominated topic sets):** **0.5** (3 topics dominated under both, 3 under one-only, 3 under neither).
- **Frac fully-dominated agreement across topics:** **0.667** (6 / 9 topics agree on dominated/not status).
- **Counterfactual hierarchy `scalar < global_mo < oracle`:** holds under **both** evaluators.
- **Counterfactual deltas (OpenAI − Google):**
  - scalar: 0
  - global_mo: +0.111 (OpenAI keeps one more topic)
  - speciated_oracle: 0
- **Spearman ρ on global Pareto rank** (all 5,655): **0.706**; same-rank fraction 2.33 %; rank-diff median 68. Among Google-F₀ genomes, only 16 % have the same OpenAI rank; among OpenAI-F₀ genomes, only 11 % have the same Google rank.

**Answers (Section M.3):**
- "Is the failure mode evaluator-specific?" — **No.** Topic-level domination disagreement is moderate (Jaccard 0.5 on fully-dominated sets, 0.667 agreement frac), and the qualitative hierarchy holds. The headline "Pareto retains 4–5/9 topics" survives evaluator swap.
- "Are evaluators interchangeable on the same responses?" — **No, at the genome level.** F₀ Jaccard 0.07 + Spearman 0.71 = "same general ordering, different exact non-dominated set." This is a **published-grade evaluator robustness caveat**, not a contradiction. §5 must explicitly say so.

**Artifacts:** `cross_evaluator_robustness.csv`, `cross_evaluator_robustness.json`, hero figure, `phase9_manifest.json`.

**Caveats / impact on later phases:**
- Phase 11 should record the **per-topic agreement string** (`both_dominated, google_only, openai_only, neither`) directly in `topic_advanced_analytics.csv` so the §5 sentence "5/9 dominated under Google, 4/9 under OpenAI, 3 dominated under both" has a single auditable source.
- No plan update needed — the M.3 row "Is this only a Perspective quirk?" already maps here, and Section H anchors match these numbers.

## Phase 11 — Deep Advanced Analytics

Goal: prove that **focusing on a single moderation metric erases multi-metric niches**, motivating speciation + multi-objective optimization for LLM red-teaming. Per-topic profile + per-axis policy survival + speciated-oracle baseline.

Outputs (`results/phase11/`):
- `topic_advanced_analytics.csv` — one row per topic.
- `single_axis_counterfactual_survival.csv` — long-format topic survival under {8 single Google axes, 13 single OpenAI axes} ∪ {global Pareto, speciated oracle}.
- `phase11_summary.json` — aggregate numbers feeding §5.
- 3 hero figures.

In [ ]:
import importlib

importlib.reload(analysis_utils)
from analysis_utils import save_phase11_artifacts

phase11_out = save_phase11_artifacts(
    rows, RESULTS_DIR / "phase11", RUN_ID, min_topic_size=MIN_TOPIC_SIZE
)
phase11_paths = phase11_out["artifacts"]
phase11_summary = phase11_out["meta"].get("summary", {})

print("Phase 11 artifacts:")
for k, v in sorted(phase11_paths.items()):
    print(f"  {k}: {v}")

display(pd.read_csv(phase11_paths["topic_advanced_analytics_csv"]).head(15))

print("\n--- summary ---")
display(pd.json_normalize(phase11_summary))

surv = pd.read_csv(phase11_paths["single_axis_counterfactual_survival_csv"])
print("\n--- counterfactual survival ---")
display(surv.sort_values(["evaluator", "topics_survived"], ascending=[True, False]))

### Phase 11 — Findings

**Hero figures (3, in `phase11/figures/`):**
- `fig11_single_axis_survival.png` — per-policy topic-retention bars (8 single Google axes + 13 single OpenAI axes + global Pareto + speciated oracle).
- `fig11_topic_profile_vs_domination.png` — per-topic axis profile heatmap with TDI (Google + OpenAI) and niche-specialization columns.
- `fig11_distinct_but_dominated.png` — TDI vs per-topic embedding silhouette scatter, point colour = niche specialization (entropy), size = topic size.

**Headline numbers (`phase11_summary.json`):**

| metric | Google | OpenAI |
|---|---|---|
| `|F₀|` (genomes on global Pareto front) | 100 | 144 |
| Topics on the global front (out of 9) | **4** | **5** |
| Topicless genomes on global front | **58 / 100** | **119 / 144** |
| Best single moderation axis (topics retained) | **threat → 5** | **illicit/violent → 6** |
| Worst single moderation axis | **toxicity → 1** | **harassment → 1** |
| Topics not recovered by **any** single axis | **3** | **1** |
| Speciated oracle | **9 / 9** | **9 / 9** |
| Mean niche-specialization (entropy of profile) | 0.588 | 0.453 |
| Mean per-topic embedding silhouette | 0.208 | 0.208 |

**Per-topic snapshot (`topic_advanced_analytics.csv`):**

| species | size | TDI(g) | TDI(o) | flip pattern | spec(g) | silh | Google axes that retain it |
|---------|------|--------|--------|---------------|---------|------|---------------------------------|
| **2421** | 150  | 0.760  | 0.907  | neither       | 0.864 (well-rounded) | **0.787** | flirtation, insult, profanity, severe_tox, sex_explicit, threat, toxicity (7 of 8) |
| 2422    | 88   | 0.977  | 0.955  | neither       | 0.563   | 0.315 | flirtation, identity_attack, threat |
| 2423    | 44   | 0.932  | 0.977  | neither       | 0.727   | 0.194 | flirtation, sex_explicit, threat |
| **2424**| 54   | 1.000  | 0.926  | google_only   | 0.626   | 0.081 | threat (only) |
| **2425**| 24   | 0.958  | 1.000  | openai_only   | 0.677   | 0.282 | flirtation, sex_explicit |
| **2426**| 61   | 1.000  | 0.967  | google_only   | 0.570   | -0.031 | threat (only) |
| 2415    | 25   | 1.000  | 1.000  | both          | 0.412   | 0.110 | **none** |
| 2427    | 6    | 1.000  | 1.000  | both          | 0.445   | -0.015 | **none** |
| 2428    | 8    | 1.000  | 1.000  | both          | 0.404   | 0.146 | **none** |

**Answers (Section M.3 + the professor's question, verbatim):**
- *"Given multiple metrics/objectives, does searching along the Pareto front of those objectives recover varying topics?"* — **No, on this run.** Global Pareto retains only 4/9 (Google) or 5/9 (OpenAI) topics, even though the speciated oracle preserves all 9.
- *"Why does single-metric optimization erase niches?"* — **Three Google topics (2415, 2427, 2428) are not recovered by ANY of the 8 single Google axes** — they are all-around moderate-scoring; no single axis "carries" them. Even Pareto over those eight axes still misses them. The dominant species 2421 (well-rounded, specialization = 0.864) is recovered by 7/8 single axes — the "front-leading" topic that any scalar-axis policy would over-represent.
- **Striking inversion:** the search itself was driven by `toxicity`, but `toxicity` is the **worst** single Google axis for topic coverage (1/9). The best single Google axis is `threat` (5/9), still worse than full multi-objective Pareto (4/9) — wait, `threat` actually beats global Pareto here at 5 vs 4. **This is interesting:** sometimes a single off-axis metric covers more *topics* than the full Pareto, because Pareto over correlated axes "double-counts" the front-leader.
- *"Why does the paper need both speciation and MO?"* — Speciation is needed because **none of the front-based or single-axis policies achieve 9/9**, only the per-topic local front (oracle) does. MO is needed because no single axis is a universal winner (best Google axis covers 5/9; best OpenAI axis covers 6/9) — combining axes via Pareto is still better than any single axis on average.

**Artifacts:** `topic_advanced_analytics.csv` (9 rows × 70+ cols), `single_axis_counterfactual_survival.csv` (23 rows: 8 Google + 13 OpenAI + 2 global_mo + 2 oracle), `phase11_summary.json`, three hero figures, `phase11_manifest.json`.

**Caveats / impact on later phases:**
- `domination_flip_pattern` is the canonical column for the §5 sentence "X topics dominated under Google only / OpenAI only / both / neither." Use these counts (3 both, 2 google_only, 1 openai_only, 3 neither) directly in the report.
- `niche_specialization_google` is **higher for the on-front winner (2421 = 0.864)** and **lower for the all-dominated topics (2415 = 0.412, 2427 = 0.445, 2428 = 0.404)**, *opposite* of the naive expectation that "specialised niches get pushed out." Here, **multi-axis well-rounded niche dominates the others on every axis simultaneously** — exactly the failure mode the professor predicted.
- **Plan-update trigger:** Section M.4.3 of the plan currently says "Some species peak on non-toxicity axes (e.g. flirtation, insult) but remain dominated in aggregate score space." Phase 11 confirms this for **2425** (peaks on flirtation + sex_explicit, dominated under OpenAI), **2424** and **2426** (recoverable only by `threat`, dominated under Google). The §5 wording in `M.7` should be updated to cite *those specific topics* and the 3-topic "no single axis recovers" anchor. We log this in the planned `M.8` plan-update step.
- Phase 10 (manifest) must list the three Phase 11 hero figures and the two Phase 11 CSVs in `ANALYSIS_MANIFEST.json` and `figures/PAPER_FIGURE_INDEX.md`.

## Phase 12 — Semantic topic labels and exemplars (NLP-readable)

For each discovered topic (species), produces:
- **TF-IDF top tokens** discriminative against the rest of the population (after deterministic redaction of harmful nouns/verbs).
- **Nearest-centroid exemplar prompts** (top-3 by cosine similarity to the topic's embedding centroid).
- **Short academic label** — fallback derived from TF-IDF tokens; an optional cached OpenAI Chat Completions call (deterministic seed) provides a 2-4 word clinical label when an API key is available.
- **Figure 5 (paper):** `phase12/figures/fig5_niche_specialization_vs_tdi_labeled.png` — niche specialization vs TDI scatter with labels overlaid (joins Phase 11 + Phase 12).

Outputs: `phase12/topic_semantic_labels.csv`, `phase12/topic_exemplars.md`, `phase12/phase12_summary.json`.

In [ ]:
import importlib

importlib.reload(analysis_utils)
from analysis_utils import save_phase12_artifacts

phase12_paths = save_phase12_artifacts(
    rows,
    out_dir=results_dir / "phase12",
    run_id=run_id,
    min_topic_size=min_topic_size,
    enable_llm=False,  # set True if OPENAI_API_KEY is available; cached output reused on rerun
)
print(json.dumps(phase12_paths.get("meta", {}), indent=2))

### Phase 12 — Findings

**Hero figure (1):** `phase12/figures/fig5_niche_specialization_vs_tdi_labeled.png` — NSI vs TDI scatter with topic labels (joins Phase 11 advanced analytics with Phase 12 semantics). Used as Figure 5 in the paper.

**Outputs (regenerated):**
- `phase12/topic_semantic_labels.csv` — `species_id`, `n_members`, `label`, `label_source ∈ {cache, api, fallback}`, `fallback_label`, `llm_label`, `top_tokens`, `top_token_scores`, `exemplar_genome_ids`, `exemplar_similarities`.
- `phase12/topic_exemplars.md` — per-topic redacted exemplar prompts (deterministic regex redactor for weapons / drugs / self-harm / violence / cyber / WMD / minors / slurs / financial crime / stalking).
- `phase12/phase12_summary.json` — diagnostics: `n_topics`, `n_label_cache_hits`, `n_label_api_calls`, `n_label_missing_key`, `n_label_errors`, `llm_enabled`, `llm_model`, `llm_seed`.
- `phase12/label_cache.json` — per-topic LLM responses keyed by SHA256(species_id + tokens + exemplars + model + seed); subsequent runs reuse this without API calls.

**Why this matters for the paper:** the §5 / §6 prose names topics in human-readable language ("violent threats", "self-harm narrative", …) instead of numeric `species_id`s; exemplars (redacted) become the appendix evidence requested by reviewers when interpreting Figure 5. Labels feed Table 1 directly.

## Phase 10 — Analysis Package

Manifest linking all artifacts, auto-generated results snippet, validation checklist vs plan anchors, paper figure index.

Optional one-shot: `run_all_remaining_phases(rows, RUN_PATH, RUN_ID)` (Phases 4–10).

In [ ]:
import importlib

importlib.reload(analysis_utils)
from analysis_utils import save_phase10_artifacts

phase10_out = save_phase10_artifacts(RESULTS_DIR, RUN_ID)
print("Phase 10 artifacts:")
for k, v in sorted(phase10_out["artifacts"].items()):
    print(f"  {k}: {v}")
print("\n--- RESULTS_SNIPPET.md ---")
print((RESULTS_DIR / "RESULTS_SNIPPET.md").read_text())

### Phase 10 — Findings

**No hero figure** — this phase only emits indices.

**Outputs (regenerated last so they include Phase 11):**
- `results/ANALYSIS_MANIFEST.json` — full file listing per phase (now also `phase11`).
- `results/RESULTS_SNIPPET.md` — auto-bullet summary for advisor review; now closes with two Phase 11 lines (best single axis per evaluator + the `n_topics_no_single_axis_recovers_*` numbers).
- `results/validation_checklist.json` — anchor vs measured values.
- `results/figures/PAPER_FIGURE_INDEX.md` — single source of truth for §4 / §5 figure references.

**Validation checklist (`results/validation_checklist.json`) — every anchor matches measured (delta = 0):**

| metric | anchor | measured |
|---|---|---|
| n_genomes | 5,655 | 5,655 |
| n_topics | 9 | 9 |
| n_f0_google | 100 | 100 |
| n_fully_dominated_topics_google | 5 | 5 |
| n_generations_collapsed_to_one | 18 | 18 |
| google_mo_topics_survived | 4 | 4 |
| openai_mo_topics_survived | 5 | 5 |
| speciated_oracle_topics_survived | 9 | 9 |
| n_topics_no_single_axis_recovers_google | 3 | 3 |
| n_topics_no_single_axis_recovers_openai | (no anchor) | 1 |

**Answers (Section M.3):** none directly — this phase is bookkeeping. It guarantees the §5 conclusion can cite a single auditable manifest + checklist for every number.

**Artifacts:** `ANALYSIS_MANIFEST.json`, `RESULTS_SNIPPET.md`, `validation_checklist.json`, `figures/PAPER_FIGURE_INDEX.md`.

**Caveats / impact:** none. The figure index now lists exactly the hero set the report should pull from (≤3 per phase + Phase 11's three deep-advanced figures). When report writing begins, point §4 / §5 at this file.